In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from treeple import RandomForestClassifier, ObliqueRandomForestClassifier
import pandas as pd
from sklearn.metrics import accuracy_score
from treeple import RandomForestClassifier
from scipy import ndimage
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import VotingClassifier
from matplotlib import pyplot as plt

In [32]:
class SillyF():
    def __init__(self):
        self.encoders = []

    def add_task(self, X, y, default_transformer_class):
        encoder = default_transformer_class
        encoder.fit(X, y)
        self.encoders.append(encoder)

    def predict(self, X, y):
        proba = 0
        for encoder in self.encoders:
            proba += encoder.predict_proba(X)
        proba = proba / len(self.encoders)
        y_pred = np.argmax(proba, axis=1)
        
        return y_pred


In [34]:
path = "D:/jhu/OneDrive - Johns Hopkins/courses/BDD/transfer/"
df = pd.read_excel(path+'Human.parcellated_thickness.xlsx')
df.head()

df_sex = pd.read_excel(path+'subjects_age_sex_data_MRI.xlsx')
df_sex.head()

X1_human = []
X2_human = []
y_human = []
IDs = set(df['sid'])
ref_IDs = set(df_sex['ID'])

# match sex and feature
for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['sid'] == subject]).reshape(-1)[2:]
        gender = list(df_sex[df_sex['ID'] == subject]['Sex'])
        sex = int(gender[0] == 'FEMALE')

        X1_human.append(list(features[:182]))  # feature1(Markov)
        X2_human.append(list(features[182:]))  # feature2(Scheafer)
        y_human.append(sex)

# switch to numpy array (easy in ML)
X1_human = np.array(X1_human)
X2_human = np.array(X2_human)
y_human = np.array(y_human)



df = pd.read_excel(path+'Macaque.parcellated_thickness.xlsx')
df.head()
df_sex = pd.read_csv(path+'uwmadison.csv')
df_sex.head()
X1_macaque = []
X2_macaque = []
y_macaque = []
IDs = set(df['participant_id'])
ref_IDs = set(df_sex['participant_id'])

for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['participant_id'] == subject]).reshape(-1)[4:]
        gender = list(df_sex[df_sex['participant_id'] == subject]['sex'])
        sex = int(gender[0] == 'F')

        X1_macaque.append(list(features[:182]))
        X2_macaque.append(list(features[182:]))
        y_macaque.append(sex)

X1_macaque = np.array(X1_macaque)
X2_macaque = np.array(X2_macaque)
y_macaque = np.array(y_macaque)

valid_indices = ~np.isnan(X1_human).any(axis=1) & ~np.isnan(X2_human).any(axis=1)
X1_human = X1_human[valid_indices]
X2_human = X2_human[valid_indices]
y_human= np.array(y_human)[valid_indices]

valid_indices = ~np.isnan(X1_macaque).any(axis=1) & ~np.isnan(X2_macaque).any(axis=1)
X1_macaque = X1_macaque[valid_indices]
X2_macaque = X2_macaque[valid_indices]
y_macaque= np.array(y_macaque)[valid_indices]

mean_X1, std_X1 = np.mean(X1_human), np.std(X1_human)
mean_X2, std_X2 = np.mean(X2_human), np.std(X2_human)
X1_human = (X1_human - mean_X1) / std_X1
X2_human = (X2_human - mean_X2) / std_X2
X_human = np.hstack((X1_human, X2_human))

mean_X1, std_X1 = np.mean(X1_macaque), np.std(X1_macaque)
mean_X2, std_X2 = np.mean(X2_macaque), np.std(X2_macaque)
X1_macaque = (X1_macaque - mean_X1) / std_X1
X2_macaque = (X2_macaque - mean_X2) / std_X2
X_macaque = np.hstack((X1_macaque, X2_macaque))

100%|███████████████████████████████████████████████████████████████████████████████| 592/592 [00:00<00:00, 910.59it/s]


In [35]:
X_train, X_test, y_train, y_test = train_test_split(X_macaque, y_macaque, test_size=0.2, random_state=42)

In [ ]:
mean_accuracies_rf = []
std_accuracies_rf = []
n_estimators = [100, 500, 1000, 2500, 5000] 

for n in n_estimators:
    accuracies = []
    kf = KFold(n_splits=5, shuffle=True, random_state=1)

    for train_index, test_index in kf.split(X_macaque):
        X_train, X_test = X_macaque[train_index], X_macaque[test_index]
        y_train, y_test = y_macaque[train_index], y_macaque[test_index]
    
        model = SillyF()
        rf=RandomForestClassifier(n_estimators=n, random_state=n)
        model.add_task(X_human, y_human, rf)
        model.add_task(X_train, y_train, rf)
        y_pred = model.predict(X_test, y_test)
        accuracy = np.sum(y_test == y_pred)
        accuracy = accuracy / len(y_test)
        accuracies.append(accuracy)
    
    mean_accuracies_rf.append(np.mean(accuracies))
    std_accuracies_rf.append(np.std(accuracies))
    
print(mean_accuracies_rf)
print(std_accuracies_rf)

In [ ]:
mean_accuracies_sporf = []
std_accuracies_sporf = []
n_estimators = [100, 500, 1000, 2500, 5000] 

for n in n_estimators:
    accuracies = []
    kf = KFold(n_splits=5, shuffle=True, random_state=1)

    for train_index, test_index in kf.split(X_macaque):
        X_train, X_test = X_macaque[train_index], X_macaque[test_index]
        y_train, y_test = y_macaque[train_index], y_macaque[test_index]
    
        model = SillyF()
        sporf=ObliqueRandomForestClassifier(n_estimators=n, random_state=n)
        model.add_task(X_human, y_human, sporf)
        model.add_task(X_train, y_train, sporf)
        y_pred = model.predict(X_test, y_test)
        accuracy = np.sum(y_test == y_pred)
        accuracy = accuracy / len(y_test)
        accuracies.append(accuracy)
    
    mean_accuracies_sporf.append(np.mean(accuracies))
    std_accuracies_sporf.append(np.std(accuracies))
print(mean_accuracies_sporf)
print(std_accuracies_sporf)

In [ ]:
#%% multi

n_estimators = [100, 500, 1000, 2500, 5000] 
mean_accuracies2 = []
std_accuracies2 = []

for n in n_estimators:
    accuracies = []
    kf = KFold(n_splits=5, shuffle=True, random_state=1)

    for train_index, test_index in kf.split(X_macaque):
        x_train, x_test = X_macaque[train_index], X_macaque[test_index]
        y_train, y_test = y_macaque[train_index], y_macaque[test_index]
    
        x_train = np.vstack((X_human, x_train))
        y_train = np.concatenate((y_human, y_train))

        clf = ObliqueRandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=n)
        clf.fit(x_train, y_train)
        y_pred = clf.predict(x_test)       

        accuracy = accuracy_score(y_test, y_pred)
        accuracies.append(accuracy)
    
    mean_accuracies2.append(np.mean(accuracies))
    std_accuracies2.append(np.std(accuracies))

print(mean_accuracies2)
print(std_accuracies2)